In [183]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [36]:
words = open('names.txt', 'r').read().splitlines()
print(words[:5])
print(len(words))

['emma', 'olivia', 'ava', 'isabella', 'sophia']
32033


In [214]:
# Choose n-gram context window
n = 4

# Build vocab of itos and stoi
chars = sorted(list(set(''.join(words))))

# Vocabulary and mappings between string and integer
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

vocab_size = len(stoi)
embedding_size = 10


# Dataset of X -> Y mappings
X, Y = [], []

"""
For each name, keep a sliding window context of the last n characters.
For the first n-1 characters, populate the empty places with '.'
E.g. for n = 4, the sliding context windows for "cedric" would be
. . . -> c
. . c -> e
. c e -> d
c e d -> r
e d r -> i
d r i -> c
""";

for word in words:
    context = [0] * (n-1)
    for char in word:
        ix = stoi[char]
        X.append (context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [216]:
len(X), len(Y)

(196113, 196113)

In [313]:
"""
Now we have the dataset we need to start training. Lets build our model.

We're building our model following Bengio et al. 2003 (A Neural Probabilistic Language Model)
where instead of words, we will have a vocab of 27 characters (including the special char '.').

We will choose a context window n for each word.
Each input x is a list of n characters that maps to an ouptut y which is the (n+1)-th character in the word.

Each character is one-hot encoded and multiplied by a 27 x 10 lookup table C to be embedded in a 10 imensional vector space.

We will pass our inputs of chosen context window through a MLP of 4 layers of 100 neurons.
Each layer will have a linear transformation wx + b, and non-linear activation function tanh.
With deeper layers of linear transformation, we will have increasing variance. 
This will lead to a greater saturation of the tails of tanh, which will kill the neuron.
We need to combat this by normalizing the distribution of activations before passing them to tanh.

There are a few ways to do this:

Kaiming initialization, whereby we construct the standard deviation of the sampling distribution that 
weights are drawn from initialization by multiplying by "empirical gain" and dividing it by the square root of fan-in (number of inputs)

The "empirical gain" counteracts for the lost variance due to the squashing of the nonlinearity
For our tanh, this constant happens to be 5/3 (due to some math that figured out how much tanh squashes).
For a nonllinearity like ReLU (which squashes all negatives), it would be sqrt(2) to compensate for all the lost negatives.

Since output variance = input variance * fan-in, our target weight variance is 1/fan-in to cancel the linear inflation due to fan-in.
To get our target weight standard deviation, we square root by the target weight variance, which is sqrt(1/fan-in) = 1/sqrt(fan-in).

So we multiply by 5/3 which is the empirical gain to add back the variance lost due to tanh and divide by
sqrt(fan-in) to counter variance inflation due to fan-in to give us our Kaiming initialization for this model.
Therefore, the standard deviation of the sampling distribution that we draw our weights from should be k_std = (5/3)/sqrt(fan-in)

This solves the weight variance problem at initialization. However, as traiing progresses, weights will change and alter the distribution.
We need a dynamic method to correct this shifting distribution. Following Ioffe et al. 2015 (Batch Normalization...), we can use the
batch normalization method to dynamically correct the shifting distribution at every step. 

For each batch of inputs, we can center the mean at 0 and normalize the standard deviation of the activations.
Each activation x_i is normalized by subtracting the mean and dividing by the square root of the variance
plus a small constant epsilon to prevent division by zero if the variance is zero: x_i_hat = (x_i - mean)/sqrt(variance + epsilon)

But this creates another problem. The mean and standard deviation is batch dependent.
During training, each input is evaluated in the context of its own batch for its mean and standard deviation.

However, during inference, whereby we evaluate a single input at a time, we do not have local context from other inputs.
If we just threw the input in and let it pass by the activation layers unnormalized, it would result in garbage since 
the model's parameters were tuned on normalized activations.

So we keep track of a global "running mean" and "running variance" that we update during training
which we can use to normalize single input inferences that work with the parameters of the model.

The purpose of normalizing the inputs is to let it start off with a "baseline" normal input to the nonlinearity
so it doesn't get stuck at the "tails" of the activation function. Over time, as the network learns,
it will update some scaling paramter gamma and shifting paramter beta, slowly shifting it to the nonlinear tails of the nonlinearity.
If we didn't normalize it first, the variance could instantly throw it at tails of the nonlinearity and never return. 
""";



In [492]:
# n-1 input chars * C embeddings -> flatten -> fan-in -> 100 neurons x 5 layers -> 100 fan-in -> 27 logits per char -> softmax -> sample nth char

# Initialize lookup table for 27 characters with 10 embeddings each
C = torch.randn(vocab_size, embedding_size)

# Lets initialize our first layer. Remember that fan-in should be n-1 * embeddings.
fan_in = (n-1) * embedding_size

# Each of the 100 neurons will accept a fan-in of 27, and add one bias.
W1 = torch.randn(fan_in, 100)
B1 = torch.randn(100)

# Final layer of 100 x 27 to create logits for each of our 27 possible characters
W2 = torch.randn(100, vocab_size)
B2 = torch.randn(vocab_size)

parameters = [C, W1, B1, W2, B2]

for p in parameters:
    p.requires_grad = True

In [468]:
# First, one-hot encode all characters.

Xenc = []
Yenc = []

for x, y in zip(X, Y):

    xenc = []
    for c in x:
        cxenc = [0] * 27
        cxenc[c] = 1
        xenc.append(cxenc)
    
    cyenc = [0] * 27
    cyenc[y] = 1

    Xenc.append(xenc)
    Yenc.append(cyenc)

Xenc = torch.tensor(Xenc).float()
Yenc = torch.tensor(Yenc).float()

Xenc.shape, Xenc.dtype, Yenc.shape, Yenc.dtype

(torch.Size([196113, 3, 27]),
 torch.float32,
 torch.Size([196113, 27]),
 torch.float32)

In [488]:
# Now lets start our try a forward pass. Mat mul the embeddings with W1 and then add B1, and then run it through a tanh() nonlinearity.

# Lets sample a "minibatch", a random set of examples from the entire dataset
batch_size = 1000
batch = torch.randint(0, Xflat.shape[0], (batch_size,)) # Chooses 32 random integers between 0 and our dataset size

X_batch = []
Y_batch = []
for i in batch:
    
    X_batch.append(C[X[i.item()]])
    Y_batch.append(Yenc[i.item()])
    
X_batch = torch.stack(X_batch)
Y_batch = torch.stack(Y_batch)


X_batch = X_batch.view(-1, 30)
forward = torch.tanh(X_batch @ W1 + B1)
forward.shape

torch.Size([1000, 100])

In [489]:
# Now lets mat mul our neuron outputs into one final layer of 100 x 27 to create logits for each of our 27 possible characters
logits = forward @ W2 + B2

In [490]:
# And now lets calculate the loss using cross_entropy()

criterion = torch.nn.CrossEntropyLoss()
loss = criterion(logits, Y_batch.float())

loss

tensor(2.5277, grad_fn=<DivBackward1>)

In [491]:
# Now lets do a backward pass, calculating the loss with respect to each of the parameters  and then updating each parameter
loss.backward()

for p in parameters:
    p.data += -0.01 * p.grad
    p.grad.zero_() # Zero the gradients to reset for next backward pass

In [473]:
# Forward again and calculate loss
batch = torch.randint(0, Xflat.shape[0], (batch_size,)) # Chooses 32 random integers between 0 and our dataset size

X_batch = []
Y_batch = []
for i in batch:
    X_batch.append(C[X[i]])
    Y_batch.append(Yenc[i])
    
X_batch = torch.stack(X_batch)
Y_batch = torch.stack(Y_batch)

X_batch = X_batch.view(-1, 30)

forward = torch.tanh(X_batch @ W1 + B1)
logits = forward @ W2 + B2

criterion = torch.nn.CrossEntropyLoss()
loss = criterion(logits, Y_batch.float())

loss

tensor(16.5397, grad_fn=<DivBackward1>)

In [493]:
# Create a loop

criterion = torch.nn.CrossEntropyLoss()

def train(loops):
    
    for loop in range(loops):
        # Forward pass
        batch = torch.randint(0, Xflat.shape[0], (batch_size,)) # Chooses 32 random integers between 0 and our dataset size
        
        X_batch = []
        Y_batch = []
        for i in batch:
            X_batch.append(C[X[i]])
            Y_batch.append(Yenc[i])
            
        X_batch = torch.stack(X_batch)
        Y_batch = torch.stack(Y_batch)
        
        X_batch = X_batch.view(-1, 30)
        
        forward = torch.tanh(X_batch @ W1 + B1)
        logits = forward @ W2 + B2
    
        loss = criterion(logits, Y_batch.float())
        
        if loop % 100 == 0:
            print(loop, loss)
        
        # Backward pass
    
        loss.backward()
        
        for p in parameters:
            p.data += -0.01 * p.grad
            p.grad.zero_() # Zero the gradients to reset for next backward pass

In [494]:
train(10000)

0 tensor(18.6790, grad_fn=<DivBackward1>)
100 tensor(13.1711, grad_fn=<DivBackward1>)
200 tensor(11.4521, grad_fn=<DivBackward1>)
300 tensor(10.3469, grad_fn=<DivBackward1>)
400 tensor(9.0878, grad_fn=<DivBackward1>)
500 tensor(8.6632, grad_fn=<DivBackward1>)
600 tensor(8.0942, grad_fn=<DivBackward1>)
700 tensor(7.5960, grad_fn=<DivBackward1>)
800 tensor(7.1141, grad_fn=<DivBackward1>)
900 tensor(6.7948, grad_fn=<DivBackward1>)
1000 tensor(6.5995, grad_fn=<DivBackward1>)
1100 tensor(6.1865, grad_fn=<DivBackward1>)
1200 tensor(6.1275, grad_fn=<DivBackward1>)
1300 tensor(5.8541, grad_fn=<DivBackward1>)
1400 tensor(5.7100, grad_fn=<DivBackward1>)
1500 tensor(5.7598, grad_fn=<DivBackward1>)
1600 tensor(5.4643, grad_fn=<DivBackward1>)
1700 tensor(5.3864, grad_fn=<DivBackward1>)
1800 tensor(5.4417, grad_fn=<DivBackward1>)
1900 tensor(5.0836, grad_fn=<DivBackward1>)
2000 tensor(5.1114, grad_fn=<DivBackward1>)
2100 tensor(4.8600, grad_fn=<DivBackward1>)
2200 tensor(4.9564, grad_fn=<DivBackward